# 第7周 · 作业（预计约7小时）

| 部分 | 内容 | 建议时间 |
| --- | --- | --- |
| A | 补全 micrograd：`__pow__`、`relu`、`exp`、减法与除法 | 3 小时 |
| B | 通过全部梯度检查（数值差分 + PyTorch 对照） | 1 小时 |
| C | 月牙数据：隐藏层宽度 4 vs 16 | 1.5 小时 |
| D | 选做（挑战）：NumPy 矩阵版两层网络 | 1.5 小时 |
| E | 反思 | 15 分钟 |

**约定**：本周作业不用 AI 编程助手写代码，只让它解释报错信息。做完后加一次 5 分钟的**自我讲解**：任选一个运算，不看注释，把它的 `_backward` 为什么这么写讲给自己听（能讲给学习伙伴听更好）。

本周作业的蓝本是 Andrej Karpathy 的 [micrograd](https://github.com/karpathy/micrograd)（MIT 许可）：A 部分补全的正是它 `engine.py` 里的运算（外加视频里有、仓库里没有的 `exp`）。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import math
import random

import numpy as np
import matplotlib.pyplot as plt
from zero2researcher.checks import week07 as ck
from zero2researcher.checks import report

## A · 补全 micrograd

下面是完整的类框架：讲义里写过的 `__add__`、`__mul__`、`tanh`、`backward` 已就位，你来补全剩下的运算。分两类：

**要写新导数的**（都在 `_backward` 闭包里，记住 `+=`）：

- `__pow__(self, other)`：$x^n$。`other` 只支持 int/float（和 micrograd 一样用 `assert` 挡住别的类型）。幂法则：$\dfrac{d}{dx}x^n = n\,x^{n-1}$；
- `relu`：$\max(0, x)$。前向：负数变 0，正数原样；反向：正半边放行 `out.grad`，负半边传 0；
- `exp`：$e^x$。它的导数就是它自己——前向已算出的 `out.data` 直接可用。

**不写新导数、用已有运算拼出来的**（各一行，反向传播自动顺着已有运算走）：

- `__neg__` = `self * -1`；`__sub__` = `self + (-other)`；`__rsub__` = `other + (-self)`；
- `__truediv__`：除法经乘方实现，`self * other ** -1`；`__rtruediv__` = `other * self ** -1`。

In [ ]:
class Value:
    """完整版标量自动微分节点（micrograd engine.py 的结构 + tanh/exp）"""

    def __init__(self, data, _children=(), _op=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    # ---------- 讲义里写过的部分（已给出） ----------
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")

        def _backward():
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo, visited = [], set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    # ---------- 你来补全：要写新导数的三个运算 ----------
    def __pow__(self, other):
        assert isinstance(other, (int, float)), "指数只支持 int/float"
        # 在这里写你的代码
        raise NotImplementedError

    def relu(self):
        # 在这里写你的代码
        raise NotImplementedError

    def exp(self):
        # 在这里写你的代码
        raise NotImplementedError

    # ---------- 你来补全：由已有运算拼出来的五个 ----------
    def __neg__(self):                    # -v
        # 在这里写你的代码
        raise NotImplementedError

    def __sub__(self, other):             # a - b
        # 在这里写你的代码
        raise NotImplementedError

    def __rsub__(self, other):            # 1.0 - v
        # 在这里写你的代码
        raise NotImplementedError

    def __truediv__(self, other):         # a / b
        # 在这里写你的代码
        raise NotImplementedError

    def __rtruediv__(self, other):        # 2.0 / v
        # 在这里写你的代码
        raise NotImplementedError


In [ ]:
# 新补的运算
ck.value_pow(Value)
ck.value_relu(Value)
ck.value_exp(Value)
ck.value_sub(Value)
ck.value_div(Value)

# 回归检查：补全过程中没把讲义里写过的部分改坏
ck.value_add(Value)
ck.value_mul(Value)
ck.value_tanh(Value)
ck.gradient_reuse(Value)

**试一试（不计分）**：视频 1:30 附近的玩法——用你的 `exp` 和除法把 tanh 拼出来：

$$\tanh(x) = \frac{e^{2x} - 1}{e^{2x} + 1}$$

backward 之后，梯度应与内置 `tanh` 一字不差（同一条链式法则，走了更长的一条图而已）。

In [ ]:
# 在这里写你的代码
raise NotImplementedError


## B · 通过全部梯度检查

A 部分的 `ck` 检查对照的是课程的参考实现；现在换**不依赖任何人实现**的数值差分，把每个新运算系统地查一遍（CS231n 口径：中心差分 $h=10^{-5}$；相对误差 $<10^{-7}$ 很好，$\sim 10^{-4}$ 只对带 kink 的 relu 可接受，$>10^{-2}$ 基本错了）。

把下表跑出来：对每一行，解析梯度走你的 `backward`，数值梯度走中心差分，报告相对误差。注意 relu 的检查点要**避开 0 附近**——kink 处数值梯度本身失真，那不是你实现的错。

还要注意表里每个新运算后面都**再乘了一个常数**：如果被测运算就是整个表达式的最后一个节点，它收到的 `out.grad` 恒为 1，`_backward` 里忘了乘 `out.grad` 也照样"通过"——检查用的表达式要让被测运算处在链条的**中间**。

In [ ]:
def numerical_grad(f, x, h=1e-5):
    """中心差分（第6周与本周练习写过，这里直接给出）"""
    return (f(x + h) - f(x - h)) / (2 * h)


def rel_error(g1, g2):
    return abs(g1 - g2) / max(abs(g1), abs(g2), 1e-12)


# 每行：名字、用 Value 搭的表达式、同一表达式的纯 Python 版、检查点 x0
# 每个新运算后面都再乘一个常数：让它收到的 out.grad ≠ 1，"_backward 忘乘 out.grad"才会露馅
cases = [
    ("pow",   lambda v: v ** 3 * 2.0,             lambda x: 2 * x ** 3,            1.7),
    ("relu+", lambda v: (v * 2.0).relu() * 3.0,   lambda x: 3 * max(0.0, 2 * x),   0.9),
    ("relu-", lambda v: (v * 2.0).relu() * 3.0,   lambda x: 3 * max(0.0, 2 * x),  -0.9),
    ("exp",   lambda v: v.exp() * 2.0,            lambda x: 2 * math.exp(x),       0.3),
    ("div",   lambda v: 2.0 / (v + 3.0),          lambda x: 2.0 / (x + 3.0),       1.2),
    ("sub",   lambda v: 1.0 - v * v,              lambda x: 1.0 - x * x,           0.6),
]

all_ok = True
for name, f_value, f_float, x0 in cases:
    # 在这里写你的代码
    raise NotImplementedError
    err = rel_error(analytic, numeric)
    all_ok = all_ok and (err < 1e-6)
    print(f"{name:5s}  x0={x0:5.1f}   解析 {analytic:12.8f}   数值 {numeric:12.8f}   相对误差 {err:.2e}")

report("全部运算的数值梯度检查", all_ok, "上表中有相对误差 ≥ 1e-6 的行",
       "逐行对照 A 部分的 _backward；relu 在 x0=-0.9 处解析和数值都应是 0")

最后一道对照：PyTorch。用一个把新运算都用上的表达式

$$g(x, y) = \frac{e^{x}}{y} + \mathrm{relu}(x \cdot y - x^3)$$

在 $x=0.5,\ y=1.5$ 处，两边梯度的相对误差应 $<10^{-6}$（torch 侧记得 `dtype=torch.float64`，取梯度用 `.grad.item()`）。

检查点也要挑：这里 $xy - x^3 = 0.625 > 0$，relu 处在"放行"的一侧，relu 和 $x^3$ 的反向都会被对照到。如果选的点让 relu 的输入为负（比如 $y=-1.5$），relu 这一项的梯度恒为 0，等于只检查了 exp 和除法。

In [ ]:
import torch

# 在这里写你的代码
raise NotImplementedError
ck.torch_agreement(rel)

## C · 月牙数据：隐藏层宽度 4 vs 16

讲义里用 `MLP(2, [8, 8, 1])` 训练了 150 个点；现在做一个**宽度对比实验**：同一份数据、同一套训练配置，只把两个隐藏层的宽度从 4 换成 16（第6周的"一次只改一个东西"）。

- 数据：`make_moons(n_samples=100, noise=0.1, random_state=42)`——和 micrograd 的 demo 一样用 100 个点（宽网络纯 Python 训练较慢，约 1 分钟）；标签变 ±1；
- 模型：`MLP(2, [w, w, 1])`，$w \in \{4, 16\}$；每次训练前 `random.seed(42)` 控制初始化；
- 损失与优化照讲义：hinge + L2（$\alpha=10^{-4}$），全批量 SGD 100 步，学习率 1.0 → 0.1 线性衰减；
- 记录每步的损失与准确率，画两组训练曲线，报告两个最终准确率。

`Neuron / Layer / MLP` 与讲义相同（micrograd 的 nn.py 结构），直接给出：

In [ ]:
class Module:
    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0.0

    def parameters(self):
        return []


class Neuron(Module):
    def __init__(self, nin, nonlin=True):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0.0)
        self.nonlin = nonlin

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh() if self.nonlin else act

    def parameters(self):
        return self.w + [self.b]


class Layer(Module):
    def __init__(self, nin, nout, **kwargs):
        self.neurons = [Neuron(nin, **kwargs) for _ in range(nout)]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]


class MLP(Module):
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i + 1], nonlin=(i != len(nouts) - 1))
                       for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [ ]:
from sklearn.datasets import make_moons

X_moons, y01 = make_moons(n_samples=100, noise=0.1, random_state=42)
y_moons = 2 * y01 - 1          # 0/1 → -1/+1


def train_mlp(widths, steps=100, alpha=1e-4):
    """训练 MLP(2, widths)，返回 (history, 最终训练准确率)；history 是每步 (loss, acc)"""
    random.seed(42)
    model = MLP(2, widths)
    print(f"MLP(2, {widths})：{len(model.parameters())} 个参数")
    history = []
    # 在这里写你的代码
    raise NotImplementedError
    return history, history[-1][1]


hist4, acc4 = train_mlp([4, 4, 1])
print(f"宽度 4 最终：loss {hist4[-1][0]:.4f}   训练 acc {acc4:.3f}\n")
hist16, acc16 = train_mlp([16, 16, 1])
print(f"宽度 16 最终：loss {hist16[-1][0]:.4f}   训练 acc {acc16:.3f}")
ck.mlp_accuracy(acc16)

In [ ]:
# 在这里写你的代码
raise NotImplementedError


**问题**（自由观察，4–6 句话）：两组曲线有什么差异？宽网络一定更好吗？从三个角度谈：参数个数（37 vs 337）与训练时间；最终损失与准确率；以及**换一个 `random.seed` 重跑**结果会不会变（时间允许就试 seed 0）。

（在这里回答）

## D · 选做（挑战）：NumPy 矩阵版两层网络

micrograd 逐标量建图，优雅但慢。深度学习框架真正的做法是**整块矩阵一起算**（CS231n 作业1的风格）。用纯 NumPy 实现一个两层网络，在同一份月牙数据上训练：

$$Z_1 = X W_1 + b_1, \qquad H = \tanh(Z_1), \qquad s = H\, w_2 + b_2$$

损失仍是 hinge 的均值形式（这次自己推矩阵梯度）：$\ell = \frac{1}{n}\sum_i \max(0,\ 1 - y_i s_i)$。

反向传播就是链式法则的矩阵版，逐步给出（$\odot$ 是逐元素乘）：

- 记 $m_i = \mathbb{1}[\,1 - y_i s_i > 0\,]$（哪些样本的 hinge 还"活着"），则 $\dfrac{\partial \ell}{\partial s_i} = -\dfrac{m_i\, y_i}{n}$；
- 输出层：$\dfrac{\partial \ell}{\partial w_2} = H^\top \dfrac{\partial \ell}{\partial s}$，$\quad\dfrac{\partial \ell}{\partial b_2} = \sum_i \dfrac{\partial \ell}{\partial s_i}$；
- 传回隐层：$\dfrac{\partial \ell}{\partial H} = \dfrac{\partial \ell}{\partial s}\, w_2^\top$（外积），过 tanh：$\dfrac{\partial \ell}{\partial Z_1} = (1 - H^2) \odot \dfrac{\partial \ell}{\partial H}$；
- 第一层：$\dfrac{\partial \ell}{\partial W_1} = X^\top \dfrac{\partial \ell}{\partial Z_1}$，$\quad\dfrac{\partial \ell}{\partial b_1} = \sum_i \dfrac{\partial \ell}{\partial Z_{1,i}}$。

三个要求：① 实现 `loss_and_grads`；② 用**数值梯度检查**验证每个参数（CS231n 的标准仪式——矩阵逐元素做差分太慢，抽查几个元素即可，`check_param` 已给出）；③ 全批量梯度下降训练 200 步（lr = 0.5），报告训练准确率，体会矩阵版和 A/C 部分纯 Python 的速度差距。

In [ ]:
def check_param(params, i, X, y, n_checks=5, h_step=1e-5, seed=0):
    """抽查 params[i] 的若干元素：中心差分 vs 解析梯度，返回最大相对误差（已给出）"""
    rng_c = np.random.default_rng(seed)
    _, grads = loss_and_grads(params, X, y)
    p_flat = params[i].reshape(-1)              # 视图：改它就是改原参数
    g_flat = np.asarray(grads[i]).reshape(-1)
    worst = 0.0
    for j in rng_c.choice(p_flat.size, size=min(n_checks, p_flat.size), replace=False):
        orig = p_flat[j]
        p_flat[j] = orig + h_step
        lp, _ = loss_and_grads(params, X, y)
        p_flat[j] = orig - h_step
        lm, _ = loss_and_grads(params, X, y)
        p_flat[j] = orig                        # 检查完放回去
        numeric = (lp - lm) / (2 * h_step)
        worst = max(worst, abs(numeric - g_flat[j]) / max(abs(numeric), abs(g_flat[j]), 1e-12))
    return worst


def loss_and_grads(params, X, y):
    """前向 + 反向。params = [W1, b1, w2, b2]（全是 ndarray，b2 是长度 1 的数组）；
    y 是 ±1 的数组。返回 (loss, grads)，grads 与 params 同形状同顺序。"""
    W1, b1, w2, b2 = params
    n = len(X)
    # 在这里写你的代码
    raise NotImplementedError
    return loss, (dW1, db1, dw2, db2)

In [ ]:
rng = np.random.default_rng(42)
hidden = 16
params = [rng.uniform(-1, 1, size=(2, hidden)), np.zeros(hidden),
          rng.uniform(-1, 1, size=hidden), np.zeros(1)]

# 在这里写你的代码
raise NotImplementedError
ck.mlp_accuracy(acc_numpy)

## E · 反思（2–3句话）

写下你调梯度时最卡的一个瞬间：当时的症状是什么（哪个检查红了、数字差多少），你靠什么线索定位到那一行？

---

**预告（第8周）**：PyTorch 的 autograd 就是**张量版的 micrograd**——`Value` 换成 `Tensor`、每个标量换成一整块矩阵，`loss.backward()` 背后还是同一套拓扑排序与 `_backward`。你将写出第一个真正的 PyTorch 训练循环，在 6 万张手写数字上复刻本周做的一切。

（在这里写）